# Unity Catalog ABAC policies

This notebook configures governed-tag-based attribute-based access control (ABAC) for the SalesLT and SalesJSON Gold schemas. Run it with an administrative identity that can assign governed tags and create functions and policies; it is not intended to be executed by a regular developer or analyst identity.

The `environment` widget accepts `dev` or `prod` and selects the matching `saleslt_<env>` and `salesjson_<env>` catalogs. The account-level governed tag `data_classification` must already exist with the allowed values `pii_email` and `geo_country`. Policies use those values to discover protected columns through `has_tag_value`, rather than relying only on column names.

In [0]:

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "saleslt_catalog": "saleslt_dev",
        "salesjson_catalog": "salesjson_dev"
    },
    "prod": {
        "saleslt_catalog": "saleslt_prod",
        "salesjson_catalog": "salesjson_prod"
    }
}

env = config[environment]

saleslt_catalog = env["saleslt_catalog"]
salesjson_catalog = env["salesjson_catalog"]

print("=" * 60)
print("UNITY CATALOG ABAC CONFIGURATION")
print("=" * 60)
print(f"Environment       : {environment}")
print(f"SalesLT catalog   : {saleslt_catalog}")
print(f"SalesJSON catalog : {salesjson_catalog}")
print("=" * 60)

## Analyst email masking

The next cells tag `saleslt_<env>.gold.sales_by_customer.customer_email` as `data_classification = pii_email`, create the `mask_customer_email` UDF in the selected SalesLT catalog, and create the schema policy `mask_pii_email_for_analysts`. The policy applies the UDF to tagged columns for `grp-dbx-analysts`.

The masking UDF preserves `NULL`, returns `***` for values without `@`, and otherwise exposes only the first character and domain (for example, `j***@example.com`). In validation queries, an admin or developer should see the original email, while an analyst should see the masked value.

In [0]:


customer_gold = (
    f"{saleslt_catalog}.gold.sales_by_customer"
)

spark.sql(f"""
ALTER TABLE {customer_gold}
ALTER COLUMN customer_email
SET TAGS (
    'data_classification' = 'pii_email'
)
""")

print(
    f"PII governed tag applied to "
    f"{customer_gold}.customer_email"
)

In [0]:
mask_function = (
    f"{saleslt_catalog}.gold.mask_customer_email"
)

spark.sql(f"""
CREATE OR REPLACE FUNCTION {mask_function}(email STRING)
RETURNS STRING
RETURN
    CASE
        WHEN email IS NULL
            THEN NULL

        WHEN email NOT LIKE '%@%'
            THEN '***'

        ELSE CONCAT(
            LEFT(email, 1),
            '***@',
            SUBSTRING_INDEX(
                email,
                '@',
                -1
            )
        )
    END
""")

print(
    f"Masking function created: "
    f"{mask_function}"
)

In [0]:

spark.sql(f"""
CREATE OR REPLACE POLICY mask_pii_email_for_analysts
ON SCHEMA {saleslt_catalog}.gold

COMMENT
    'Masks customer email columns classified as PII for the analyst group.'

COLUMN MASK {mask_function}

TO `grp-dbx-analysts`

FOR TABLES

MATCH COLUMNS
    has_tag_value(
        'data_classification',
        'pii_email'
    ) AS pii_email_column

ON COLUMN pii_email_column
""")

print(
    "ABAC column mask policy created successfully."
)

## Analyst country row filter

The next cells tag `salesjson_<env>.gold.customer_sales_summary.country` as `data_classification = geo_country`, create the Boolean `filter_analyst_country` UDF in the selected SalesJSON catalog, and create the schema policy `filter_country_for_analysts`. The policy passes the tagged country column to the UDF for `grp-dbx-analysts`.

The row-filter UDF returns `TRUE` only for `Costa Rica`; all other values, including `NULL`, return `FALSE` and are hidden. In validation queries, an admin or developer should see all countries, while an analyst should see only Costa Rica rows.

In [0]:

customer_sales_gold = (
    f"{salesjson_catalog}.gold.customer_sales_summary"
)

spark.sql(f"""
ALTER TABLE {customer_sales_gold}
ALTER COLUMN country
SET TAGS (
    'data_classification' = 'geo_country'
)
""")

print(
    f"Geographic governed tag applied to "
    f"{customer_sales_gold}.country"
)

In [0]:
country_filter_function = (
    f"{salesjson_catalog}.gold.filter_analyst_country"
)

spark.sql(f"""
CREATE OR REPLACE FUNCTION {country_filter_function}(country STRING)
RETURNS BOOLEAN
RETURN COALESCE(
    country = 'Costa Rica',
    FALSE
)
""")

print(
    f"Row filter function created: "
    f"{country_filter_function}"
)

In [0]:
spark.sql(f"""
CREATE OR REPLACE POLICY filter_country_for_analysts
ON SCHEMA {salesjson_catalog}.gold

COMMENT
    'Restricts analyst access to Costa Rica customer sales records.'

ROW FILTER {country_filter_function}

TO `grp-dbx-analysts`

FOR TABLES

MATCH COLUMNS
    has_tag_value(
        'data_classification',
        'geo_country'
    ) AS country_column

USING COLUMNS (
    country_column
)
""")

print(
    "ABAC row filter policy created successfully."
)

## Policy validation and evidence

Use `SHOW POLICIES` to confirm that both schema policies exist and `DESCRIBE POLICY` to inspect their principals, functions, and governed-tag match expressions. For DEV, run:

```sql
SHOW POLICIES ON SCHEMA saleslt_dev.gold;
DESCRIBE POLICY mask_pii_email_for_analysts ON SCHEMA saleslt_dev.gold;
SHOW POLICIES ON SCHEMA salesjson_dev.gold;
DESCRIBE POLICY filter_country_for_analysts ON SCHEMA salesjson_dev.gold;
```

Use the corresponding `_prod` catalogs when the widget is set to `prod`. Validate data access separately as an admin/developer and as a member of `grp-dbx-analysts`. Store screenshots of policy configuration and both identity-specific query results under `evidence/Security/ABAC`.